# 1. Data Acquisition (25%) (Manual)
by William Perktold

Contains code copied from demos of IFT 6758

October 2026


Downloads NHL play-by-play data for both the regular season and playoffs


In [62]:
# This cell is copied from Demo 2

import json
from pathlib import Path

import requests

"https://api-web.nhle.com/v1/schedule/now"
response = requests.get("https://api-web.nhle.com/v1/schedule/now", timeout=30)


In [196]:
# Code by William Perktold
# uses snippets from Demo 2


class NHLyear:
    def __init__(self, year, filepath):
        # initialises the object
        # todo: verify year is int or str representing year between 1900-2300 approx ?
        self.year = str(year)  # should be the year the season starts, in str form now
        self.filepath = filepath  # should be str
        self.game_list = []


    def download(self, game_id, dir_path, verbose=False, save=False):
        # copied from demo 1
        # downloads a game

        raw_dir = Path(dir_path) / self.year
        #raw_dir.parent.mkdir(parents=True, exist_ok=True)
        raw_dir.mkdir(parents=True, exist_ok=True)
        raw_path = raw_dir / f"{game_id}.json"
        # create the year folder if necessary

        if raw_path.exists():
            if verbose:
                print("Already downloaded — reusing", raw_path.name)
            return "200", json.loads(raw_path.read_text(encoding="utf-8"))
        else:
            url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
            response = requests.get(url, timeout=30)
            if verbose:
                print("HTTP status:", response.status_code)
                print("Content type:", response.headers.get("Content-Type"))
            #response.raise_for_status()
            if str(response.status_code) != "200":
                return response.status_code, ""
            downloaded_game = response.json()

            # Check we received the requested game's event data before saving it.

            check = (str(downloaded_game["id"]) == str(game_id)) & isinstance(downloaded_game["plays"], list)
            if check:
                raw_path.write_text(json.dumps(downloaded_game, indent=2), encoding="utf-8")
                if save:
                    self.game_list.append(downloaded_game)
                if verbose:
                    print("Saved", game_id)
            else:
                print("Wrong game found?")
                print(str(downloaded_game["id"]) + "  " + game_id)

        return response.status_code, downloaded_game

    def download_all(self, save=True):
        # Downloads all matches from that year
        # With save=True, it saves the games to a list (self.game_list)
        # This will usually take several minutes

        # game_id creation

        # loop over regular season games:
        cont = True
        i = 1
        while cont:
            game_id = self.year + "02" + f"{i:04}"  # regular season game_id is [year]02[number]
            status_code, loaded_game  = self.download(game_id, self.filepath, verbose= i % 20 == 0, save=save)
            status_code = str(status_code)
            if str(status_code) != "200":
                cont = False
                if str(status_code) != "404":
                    print(f"ERROR: {status_code}")
            i += 1
        print(f"Downloaded {i} regular season games.")
        # playoff games:
        # loop on playoff rounds :
        n = 0  # number of playoff games downloaded
        for j in range(1, 5):
            # loop over the matchups :
            for k in range(1, 2 **(4-j)+1 ):
                l = 1
                # loop over the individual games:
                while True:
                    game_id = self.year + "03" + f"{j:02}" + f"{k:01}" + f"{l:01}"  # verify?
                    l += 1
                    status_code, loaded_game = self.download(game_id, self.filepath,
                                                    verbose = (l==2), save=save)
                    if str(status_code) != "200":
                        if str(status_code) != "404":
                            print(f"ERROR Status Code: {status_code}")
                        break
                    if l>10:
                        print(f"Error {l}")
                    n += 1

        print(f"Downloaded {n} playoff games.")

    def game(self, game_id, dir_path = "data/raw"):
        # returns the game .json text. Works even without load_matches() being called
        # overload function to accept other than game_id?
        raw_dir = Path(dir_path)
        raw_dir.mkdir(parents=True, exist_ok=True)

        raw_path = raw_dir / self.year / f"{game_id}.json"
        raw_path.parent.mkdir(parents=True, exist_ok=True)

        status_code, loaded_game = self.download(game_id, dir_path,  verbose=True, save=False)

        if str(status_code) == "200":
            return loaded_game
        return status_code
        #print(response.json())


In [198]:
yr2022 = NHLyear(2022, "data/raw")
yr2022.download_all(save=False)

HTTP status: 200
Content type: application/json
Saved 2022020050
HTTP status: 200
Content type: application/json
Saved 2022020100
HTTP status: 200
Content type: application/json
Saved 2022020150
HTTP status: 200
Content type: application/json
Saved 2022020200
HTTP status: 200
Content type: application/json
Saved 2022020250


KeyboardInterrupt: 

In [197]:
print(yr2022.game(2022020060))


HTTP status: 200
Content type: application/json
Saved 2022020060
{'id': 2022020060, 'season': 20222023, 'gameType': 2, 'limitedScoring': False, 'gameDate': '2022-10-20', 'venue': {'default': 'Centre Bell'}, 'venueLocation': {'default': 'Montreal', 'fr': 'Montréal'}, 'startTimeUTC': '2022-10-20T23:00:00Z', 'easternUTCOffset': '-04:00', 'venueUTCOffset': '-04:00', 'tvBroadcasts': [{'id': 131, 'market': 'H', 'countryCode': 'CA', 'network': 'TSN2', 'sequenceNumber': 112}, {'id': 33, 'market': 'H', 'countryCode': 'CA', 'network': 'RDS', 'sequenceNumber': 145}, {'id': 343, 'market': 'A', 'countryCode': 'US', 'network': 'BSAZ', 'sequenceNumber': 216}], 'gameState': 'OFF', 'gameScheduleState': 'OK', 'periodDescriptor': {'number': 3, 'periodType': 'REG', 'maxRegulationPeriods': 3}, 'awayTeam': {'id': 53, 'commonName': {'default': 'Coyotes'}, 'abbrev': 'ARI', 'score': 2, 'sog': 27, 'logo': 'https://assets.nhle.com/logos/nhl/svg/ARI_20202021-20232024_light.svg', 'darkLogo': 'https://assets.nhle.c